# 02 · Sampling from the Trained Score Network

*Code companion to an MSc dissertation on score-based generative
modelling of financial time series (Queen Mary University of London,
2026) — see the repository README for the reference.*

This notebook loads a trained checkpoint and generates synthetic return
windows by integrating the reverse VE-SDE. Three samplers are implemented in
`sampling/`:

| Sampler | Method | Noise levels | Function evaluations (NFE) | Used in the dissertation |
|---|---|---|---|---|
| `PC_sampler` | Predictor–corrector: reverse-diffusion predictor + Langevin corrector with SNR-based step size | K = 100, m = 3, r = 0.16 | 400 | Yes (stochastic sampler) |
| `ODE_sampler` | Probability-flow ODE with Heun's method and a final denoising step | K = 30 | 60 | Yes (deterministic sampler) |
| `ALD_sampler` | Annealed Langevin dynamics (MCMC at each noise level) | — | ~10,000 | No (reference implementation) |

Both samplers were applied to all
configuration–dataset combinations, generating 1000 samples of length 256 each.

**Outputs**

| Artefact | Location |
|---|---|
| Generated samples `(num_samples, L)` | `gen_data/gs_<sampler>_<run_name>.pt` |
| Sample-path figures | `figures/<name>_returns.png` |

## 0 · Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import torch

from config import data_config
from data import get_data, DailyLogReturnsData
from model import ScoreNet
from sampling import PC_sampler, ALD_sampler, ODE_sampler
from plots import plot_sample_returns

## 1 · Real data (for visual comparison)

Rebuilt with the same settings as in training so real and generated windows
are on the same standardised scale.

In [ ]:
snp_data = get_data({
    "ticker": data_config["TICKER"],
    "start":  data_config["START"],
    "end":    data_config["END"],
})
dataset = DailyLogReturnsData(
    log_returns=snp_data["log_return"],
    window_size=data_config["WINDOW_SIZE"],
    stride=data_config["STRIDE"],
    normalize=True,
)
real_samples = torch.stack([dataset[i].squeeze(0) for i in range(len(dataset))])
print(f"Real windows: {tuple(real_samples.shape)}")

## 2 · Load a trained checkpoint

The network is rebuilt from the `model_config` stored inside the checkpoint,
so any checkpoint produced by `01_training.ipynb` can be loaded without
editing `config.py`.

In [ ]:
RUN_NAME = "conf2_s1_noise10_seed0"           # <- checkpoint to sample from
CHECKPOINT = f"./checkpoints/{RUN_NAME}.pt"

checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
mc = checkpoint["model_config"]

net = ScoreNet(
    channels      = mc["CHANNELS"],
    diffusion_dim = mc["DIFFUSION_DIM"],
    num_heads     = mc["NUM_HEADS"],
    num_blocks    = mc["NUM_RES_BLOCKS"],
    kernel_size   = mc["KERNEL_SIZE"],
)
net.load_state_dict(checkpoint["model_state_dict"])
net.to(mc["DEVICE"]).eval()

input_dim = checkpoint["train_config"].get("in_feat", data_config["WINDOW_SIZE"])
n_params = sum(p.numel() for p in net.parameters())

print(f"Loaded:       {CHECKPOINT}")
print(f"Config:       {mc['CHANNELS']} channels × {mc['NUM_RES_BLOCKS']} blocks  ({n_params:,} params)")
print(f"Final loss:   {checkpoint['losses'][-1]:.5f}")
print(f"Window L:     {input_dim}")
print(f"Device:       {mc['DEVICE']}")

## 3 · Sampler configuration

All samplers integrate the reverse VE-SDE from $t = T$ ($\sigma \approx \sigma_{\max}$)
down to $t = 0$ ($\sigma \approx \sigma_{\min}$) on a grid of `noise_levels` points.
For PC, `steps` is the number of Langevin corrector iterations per level and
`snr` the target signal-to-noise ratio $r$ of the corrector step.
The ODE sampler is deterministic given `seed`; `denoise=True` applies the
optional final denoising step at $\sigma_{\min}$.

In [ ]:
NUM_SAMPLES = 1000

PC_config = dict(
    t_min=0.0, t_max=1.0,
    noise_levels=100, steps=3, snr=0.16,
    dim=input_dim, num_samples=NUM_SAMPLES, log_every=20,
)

ODE_config = dict(
    t_min=0.0, t_max=1.0,
    noise_levels=30, heun=True, denoise=True,
    dim=input_dim, num_samples=NUM_SAMPLES, log_every=10,
)

ALD_config = dict(
    t_min=1e-3, t_max=1.0,
    noise_lvls=100, steps=50,
    dim=input_dim, num_samples=NUM_SAMPLES, log_every=20,
)

## 4 · Generate and save

Select one sampler. Generated tensors are saved with a name encoding the
sampler and the source run so they can be matched to checkpoints later:
`gs_<PC|ODE|ALD>_<run_name>.pt`.

In [ ]:
SAMPLER = "PC"                                  # "PC" | "ODE" | "ALD"

if SAMPLER == "PC":
    generated_samples = PC_sampler(net, PC_config, seed=42)
elif SAMPLER == "ODE":
    generated_samples, _, _ = ODE_sampler(net, ODE_config, seed=42)
elif SAMPLER == "ALD":
    generated_samples, _, _ = ALD_sampler(net, ALD_config)
else:
    raise ValueError(f"Unknown sampler: {SAMPLER}")

OUT_PATH = f"./gen_data/gs_{SAMPLER}_{RUN_NAME}.pt"
torch.save(generated_samples.cpu(), OUT_PATH)

print(f"Generated:   {tuple(generated_samples.shape)}")
print(f"Mean / std:  {generated_samples.mean():.4f} / {generated_samples.std():.4f}")
print(f"Saved to:    {OUT_PATH}")

## 5 · Visualise sample paths

Nine generated windows alongside nine real windows on the same scale.
Visual inspection is a sanity check only; quantitative comparison is done in
`03_evaluation.ipynb`.

In [ ]:
gen_cpu = generated_samples.cpu()

plot_sample_returns(gen_cpu[:9], show_titles=False, save_fig=True,
                    title=f"gs_{SAMPLER}_{RUN_NAME}")

plot_sample_returns(real_samples[::len(real_samples) // 9][:9],
                    show_titles=False, save_fig=True, title="real_samples")